# Đọc dữ liệu cháy rừng từ OCI Object Storage

`raw-zone` chứa các file CSV để phân tích. `system-state` chỉ chứa con trỏ của pipeline. Notebook này đọc trực tiếp từ OCI, hiển thị danh sách file và xem trước một phần CSV bằng pandas; không tải toàn bộ dữ liệu vào RAM.

Mở notebook trong thư mục `crawl-data`. Tạo `crawl-data/.env` từ `crawl-data/.env.example` và điền **cặp khóa chỉ đọc của riêng bạn**. Không đưa khóa vào notebook hoặc Git.

In [ ]:
import json
import os
from contextlib import closing
from pathlib import Path

import boto3
import pandas as pd
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from IPython.display import display

env_path = next((path for path in (Path.cwd() / '.env', Path.cwd() / 'crawl-data' / '.env') if path.is_file()), None)
if env_path is None:
    raise FileNotFoundError('Tạo crawl-data/.env từ crawl-data/.env.example trước khi chạy notebook')
load_dotenv(env_path, override=False)
required = ('S3_ENDPOINT', 'S3_ACCESS_KEY', 'S3_SECRET_KEY')
missing = [name for name in required if not os.getenv(name) or os.getenv(name).startswith('replace_with_')]
if missing:
    raise ValueError(f'Thiếu cấu hình OCI trong .env: {missing}')

s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT'],
    aws_access_key_id=os.environ['S3_ACCESS_KEY'],
    aws_secret_access_key=os.environ['S3_SECRET_KEY'],
    region_name=os.getenv('S3_REGION', 'ap-singapore-1'),
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
        request_checksum_calculation='when_required',
    ),
)
data_bucket = os.getenv('DATA_BUCKET', 'raw-zone')
print(f'Đã cấu hình đọc bucket: {data_bucket}')

## 1. Liệt kê các file CSV trong `raw-zone`

Danh sách được sắp theo thời điểm tải lên, mới nhất ở đầu. Mỗi file là một lô bản ghi.

In [ ]:
objects = [
    item
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=data_bucket, Prefix='wildfire/')
    for item in page.get('Contents', [])
    if item['Key'].lower().endswith('.csv')
]
objects.sort(key=lambda item: item['LastModified'], reverse=True)
if not objects:
    raise RuntimeError(f'Chưa có file CSV trong bucket {data_bucket}')

inventory = pd.DataFrame([
    {
        'object_key': item['Key'],
        'size_mb': round(item['Size'] / 1024**2, 2),
        'uploaded_at': item['LastModified'],
    }
    for item in objects
])
print(f'Có {len(objects)} file CSV')
display(inventory.head(20))

## 2. Xem dữ liệu CSV dưới dạng bảng

Đổi `selected_key` nếu muốn đọc một file khác trong bảng trên. `nrows` giới hạn số dòng đưa vào RAM; với file một triệu dòng, chỉ tăng giới hạn khi máy đủ bộ nhớ.

In [ ]:
selected_key = objects[0]['Key']
with closing(s3.get_object(Bucket=data_bucket, Key=selected_key)['Body']) as body:
    df = pd.read_csv(body, nrows=1000)
print(f'File: {selected_key} | Đã đọc {len(df)} dòng')
display(df.head(10))

## Tùy chọn: xem con trỏ của pipeline

Chỉ người vận hành được cấp quyền đọc `system-state` mới cần chạy ô này. Thành viên phân tích dữ liệu có thể bỏ qua.

In [ ]:
try:
    response = s3.get_object(
        Bucket=os.getenv('STATE_BUCKET', 'system-state'),
        Key='wildfire_pipeline/pointer.json',
    )
    with closing(response['Body']) as body:
        state = json.load(body)
    display(pd.DataFrame([state]))
except ClientError as error:
    if error.response.get('Error', {}).get('Code') in {'AccessDenied', '403', 'NoSuchKey', '404'}:
        print('Không có quyền đọc con trỏ hoặc con trỏ chưa tồn tại; dữ liệu raw-zone vẫn dùng được.')
    else:
        raise